# 02 · Re-check of the LGP and AEZ-Ym A/Bs

**Question.** Both earlier A/Bs were decided on **one 70/30 split (test n=14)**. Do they survive proper cross-validation? Two *different* concerns: the LGP test refits Ym per arm (level blindness), while the AEZ-Ym test fits **two** free parameters against one (degrees of freedom). The original script was not kept, so this reconstructs it from source.

**How to run:** put the `planting_pipeline` folder on your Google Drive, run top-to-bottom, approve the Drive-mount and Earth-Engine prompts. Export cells start GEE tasks and return immediately; the scoring cells read the CSVs once the tasks finish (watch https://code.earthengine.google.com/tasks).

## Setup

### Stage 0 · Runtime

Installs the Earth Engine client, geemap, pandas, geopandas and scipy. `scipy` is the one that matters
here: every score in this notebook is a leave-one-out cross-validation with a paired bootstrap interval,
and both come from scipy.

**Expected output.** `installed.`

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas scipy 2>/dev/null
print('installed.')

### Stage 0b · Earth Engine

`EE ready: ok`. The export cells below submit **batch tasks** and return immediately; the scoring cells
read the resulting CSVs. Between the two you have to wait, and you can close the browser while you do.
Watch the queue at code.earthengine.google.com/tasks.

**The export queue is per cloud project.** `ee-manzikye` has left batches in READY for hours. If the
tasks are not entering RUNNING within about 20 minutes, switch `PROJECT` to
`indigo-proxy-484220-q8` and resubmit rather than waiting.

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Drive

`pipeline on path: ...`. The scoring scripts read and write `Cropyield-Data/` inside this folder, so the
notebook must `chdir` here for the relative paths to resolve.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

### Scoring convention used throughout
`n` is small (6–81 zones) in every test here, and a single 70/30 split at n≈45 has a **±0.10 t/ha standard deviation — larger than any effect measured**. So every test below uses **leave-one-out CV** (each free parameter refit on n−1) plus a **paired bootstrap** CI, and reports **Spearman** alongside MAE because rank skill is invariant to the yield ceiling Ym. Reporting a single split would have produced two false positives in this round.

## Reconstruction
County-mean water-limited relyield from the CHIRPS MAM water balance at each duration, highland flag from SRTM ≥1800 m, HarvestStat multi-year county mean.

**Fidelity check:** the reconstruction reproduces the published Pearson r (0.66 / 0.51) almost exactly — which is what makes the re-scoring trustworthy.

### Stage 1 · Rebuild and re-score both earlier A/Bs

**What runs.** The original script was not kept, so this reconstructs both tests from source: the
county-mean water-limited relative yield from the CHIRPS MAM water balance at each season duration, a
highland flag from SRTM at 1800 m, and the HarvestStat multi-year county mean.

**Why they needed re-checking, for two different reasons.** The **LGP** test refits $Y_m$ per arm, so it
can mistake a level shift for skill. The **AEZ-Ym** test fits **two** free parameters against one, so it
can win simply by having more freedom. Leave-one-out with the parameters refit on $n-1$ pays for both.

**Fidelity check first.** The reconstruction reproduces the published Pearson r of 0.66 and 0.51 almost
exactly. That is what makes the re-scoring trustworthy; if it did not, nothing below would mean anything.

**Expected values.**

| Arm | Free parameters | LOO MAE | Pearson | Spearman |
|---|---|---|---|---|
| A, fixed 120 d, single $Y_m$ | 1 | 0.601 | +0.642 | +0.725 |
| B1, zone-aware LGP | 1 | 0.684 | +0.497 | +0.525 |
| B2, per-zone $Y_m$ 3.7 / 2.3 | 2 | **0.537** | **+0.780** | **+0.830** |

**Two different verdicts.**

* **LGP: confirmed, keep the fixed 120 days.** B1 loses on MAE and collapses the ranking. The rank half
  of the verdict is invariant to $Y_m$, so the refit trap never applied. Across 500 seeds B1 beats A in
  only 10 % of them, so the published result was typical rather than lucky.
* **AEZ-Ym: right decision, wrong evidence.** The MAE gain of +0.063 has an interval of
  [−0.046, +0.167], which is **not significant** once the second parameter is paid for. The out-of-sample
  **ranking** gain is real. So the highland lever is the ceiling, not the season length, but the claim
  should be made about rank skill and not about MAE.

The highland split is currently **off** in `src/cpi.py` (`YM_HIGHLAND = {}`) because the typical-year
calibration uses one ceiling per country and season. The commented values reproduce the 2024 fit.

In [ ]:
!python lgp_ym_ab_recheck.py

## Result

| arm | params | LOO MAE | Pearson | Spearman |
|---|---|---|---|---|
| A fixed 120 d, single Ym | 1 | 0.601 | +0.642 | +0.725 |
| B1 zone-aware LGP | 1 | 0.684 | +0.497 | +0.525 |
| B2 per-zone Ym (3.7/2.3) | 2 | **0.537** | **+0.780** | **+0.830** |

**LGP verdict CONFIRMED — keep fixed 120 d.** B1 loses on MAE and collapses the ranking; the rank half of the verdict is Ym-invariant so the refit trap never applied. Across 500 seeds B1 beats A in only 10% — the published result was typical, not lucky.

**AEZ-Ym: right decision, wrong evidence.** Δ MAE +0.063, CI [−0.046,+0.167] — **not significant** once the second parameter is paid for. But the out-of-sample *ranking* gain is real and substantial (Spearman 0.725 → 0.830), and that is not a DoF artifact. Keep `YM_HIGHLAND`, justify it by ranking rather than MAE.

**`lgp_ab_test_MAM.md` overstates precision.** It reports 0.568 → 0.472; across 500 seeds the means are 0.604 → 0.544 and B2 wins 77% of splits, not all. Worth softening if that table feeds the report.